# Import thư viện

In [1]:
import pandas as pd
import matplotlib as plt
import numpy as np
from sklearn.preprocessing import LabelEncoder
import json
import joblib
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split

## Đọc dữ liệu từ file csv

In [2]:
df_th = pd.read_csv('C:\\CS114\\th-public.csv')
df_ck = pd.read_csv('C:\\CS114\\ck-public.csv')
df_qt = pd.read_csv('C:\\CS114\\qt-public.csv')
df_total = pd.read_csv('C:\\CS114\\annonimized.csv')

## Tổng quan về dữ liệu

In [3]:
df_total.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 11 columns):
 #   Column                           Non-Null Count   Dtype 
---  ------                           --------------   ----- 
 0   concat('it001',`assignment_id`)  295198 non-null  object
 1   concat('it001',`problem_id`)     295198 non-null  object
 2   concat('it001', username)        295198 non-null  object
 3   is_final                         295198 non-null  int64 
 4   status                           295198 non-null  object
 5   pre_score                        295198 non-null  int64 
 6   coefficient                      295198 non-null  int64 
 7   concat('it001',`language_id`)    295198 non-null  object
 8   created_at                       295198 non-null  object
 9   updated_at                       295198 non-null  object
 10  judgement                        295198 non-null  object
dtypes: int64(3), object(8)
memory usage: 24.8+ MB


In [4]:
df_total.head()

,"concat('it001',`assignment_id`)","concat('it001',`problem_id`)","concat('it001', username)",is_final,status,pre_score,coefficient,"concat('it001',`language_id`)",created_at,updated_at,judgement
0,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,SCORE,0,100,it0012,10-09 08:02:04,10-09 08:06:58,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0..."
1,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,SCORE,0,100,it0012,10-09 08:04:41,10-09 08:04:51,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0..."
2,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,10-09 08:06:49,10-09 08:06:58,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0..."
3,90ce27571176d87961b565d5ef4b3de33ede04ac,bf96fbdc5f499538c3e2bfbec5779c8a14b0a9ff,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,10-09 08:47:52,10-09 08:48:01,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0..."
4,90ce27571176d87961b565d5ef4b3de33ede04ac,7a6e5ca470ff47c3b5048f240c4738de71010c78,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,10-09 09:19:35,10-09 09:19:45,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0..."


## Tiền xử lý dữ liệu

### Kiểm tra xem có dòng nào bị thiếu dữ liệu không

In [5]:
df_total.isna().sum()

concat('it001',`assignment_id`)    0
concat('it001',`problem_id`)       0
concat('it001', username)          0
is_final                           0
status                             0
pre_score                          0
coefficient                        0
concat('it001',`language_id`)      0
created_at                         0
updated_at                         0
judgement                          0
dtype: int64

### Đổi tên đặc trưng

In [6]:
df_total = df_total.rename(columns={
    "concat('it001', username)": "mssv",
    "concat('it001',`assignment_id`)": "assignment_id",
    "concat('it001',`problem_id`)": "problem_id",
    "pre_score": "score",
    "coefficient": "late_coef",
    "created_at": "submit_time",
    "updated_at": "judge_time",
    "concat('it001',`language_id`)": "lang_id",
    "judgement": "judgement_json"
})

### Chuyển các đặc trưng từ dạng chuỗi sang dạng số

In [7]:
df_total['code_runnable'] = (df_total['status'] == 'SCORE').astype(int)
df_total['compile_error'] = (df_total['status'] == 'Compilation Error').astype(int)
df_total['syntax_error'] = (df_total['status'] == 'Syntax Error').astype(int)
df_total['pending'] = (df_total['status'] == 'pending').astype(int)
df_total['testcase_passed_percent'] = (df_total['score'] / 100).astype(int)
df_total['submit_time'] = pd.to_datetime('2024-' + df_total['submit_time'], errors='coerce')
df_total['judge_time'] = pd.to_datetime('2024-' + df_total['judge_time'], errors='coerce')

In [8]:
df_total.head()

,assignment_id,problem_id,mssv,is_final,status,score,late_coef,lang_id,submit_time,judge_time,judgement_json,code_runnable,compile_error,syntax_error,pending,testcase_passed_percent
0,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,SCORE,0,100,it0012,2024-10-09 08:02:04,2024-10-09 08:06:58,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0...",1,0,0,0,0
1,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,SCORE,0,100,it0012,2024-10-09 08:04:41,2024-10-09 08:04:51,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0...",1,0,0,0,0
2,90ce27571176d87961b565d5ef4b3de33ede04ac,789454427dd4097a14749e3dde63346b7a8d3811,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,2024-10-09 08:06:49,2024-10-09 08:06:58,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0...",1,0,0,0,100
3,90ce27571176d87961b565d5ef4b3de33ede04ac,bf96fbdc5f499538c3e2bfbec5779c8a14b0a9ff,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,2024-10-09 08:47:52,2024-10-09 08:48:01,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0...",1,0,0,0,100
4,90ce27571176d87961b565d5ef4b3de33ede04ac,7a6e5ca470ff47c3b5048f240c4738de71010c78,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,SCORE,10000,100,it0012,2024-10-09 09:19:35,2024-10-09 09:19:45,"{""times"":[0,0,0,0,0,0,0,0,0,0],""mems"":[0,0,0,0...",1,0,0,0,100


### Xóa cột 'status' gốc

In [9]:
df_total = df_total.drop(columns=['status'])

### Encode các cột ID chuỗi thành số nguyên

In [10]:
df_encoded = df_total.copy()

# Encode các cột ID chuỗi thành số nguyên
for col in ['assignment_id', 'problem_id', 'lang_id']:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

### Trích xuất thông tin từ cột judgement_json, sau đó xóa cột đó đi

In [11]:
def extract_judgement_features(row):
    try:
        data = json.loads(row)
        times = data.get("times", [])
        mems = data.get("mems", [])
        failed = data.get("failed", [])

        n_tests = len(times)
        n_failed = len(failed)
        failed_ratio = n_failed / n_tests if n_tests > 0 else 0

        avg_time = np.mean(times) if times else 0
        max_time = np.max(times) if times else 0

        avg_mem = np.mean(mems) if mems else 0
        max_mem = np.max(mems) if mems else 0

        return pd.Series([n_tests, n_failed, failed_ratio, avg_time, max_time, avg_mem, max_mem])

    except:
        return pd.Series([0, 0, 0, 0, 0, 0, 0])

# Áp dụng trích xuất vào dataframe
df_encoded[['n_tests', 'n_failed', 'failed_ratio', 'avg_time', 'max_time', 'avg_mem', 'max_mem']] = df_encoded['judgement_json'].apply(extract_judgement_features)

df_encoded = df_encoded.drop(columns=['judgement_json'])

### Đổi tên cột hash thành mssv

In [12]:
df_ck.rename(columns={'hash': 'mssv'}, inplace=True)
df_qt.rename(columns={'hash': 'mssv'}, inplace=True)
df_th.rename(columns={'hash': 'mssv'}, inplace=True)

### Chuyển cột điểm từ dạng chuỗi thành dạng số

In [13]:
df_ck['CK'] = pd.to_numeric(df_ck['CK'], errors='coerce')
df_qt['diemqt'] = pd.to_numeric(df_qt['diemqt'], errors='coerce')
df_th['TH'] = pd.to_numeric(df_th['TH'], errors='coerce')

### Loại bỏ các dòng có cột điểm bị thiếu dữ liệu

In [14]:
df_ck = df_ck.dropna(subset=['CK'])
df_qt = df_qt.dropna(subset=['diemqt'])
df_th = df_th.dropna(subset=['TH'])

In [15]:
df_encoded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 21 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   assignment_id            295198 non-null  int64         
 1   problem_id               295198 non-null  int64         
 2   mssv                     295198 non-null  object        
 3   is_final                 295198 non-null  int64         
 4   score                    295198 non-null  int64         
 5   late_coef                295198 non-null  int64         
 6   lang_id                  295198 non-null  int64         
 7   submit_time              295198 non-null  datetime64[ns]
 8   judge_time               295198 non-null  datetime64[ns]
 9   code_runnable            295198 non-null  int64         
 10  compile_error            295198 non-null  int64         
 11  syntax_error             295198 non-null  int64         
 12  pending         

## Ghép dữ liệu các cột đặc trưng với dữ liệu điểm thông qua mssv

In [16]:
df_merged = df_encoded.merge(df_ck, on='mssv', how='left')
df_merged = df_merged.merge(df_qt, on='mssv', how='left')
df_merged = df_merged.merge(df_th, on='mssv', how='left')

### kiểm tra dữ liệu sau khi ghép

In [17]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   assignment_id            295198 non-null  int64         
 1   problem_id               295198 non-null  int64         
 2   mssv                     295198 non-null  object        
 3   is_final                 295198 non-null  int64         
 4   score                    295198 non-null  int64         
 5   late_coef                295198 non-null  int64         
 6   lang_id                  295198 non-null  int64         
 7   submit_time              295198 non-null  datetime64[ns]
 8   judge_time               295198 non-null  datetime64[ns]
 9   code_runnable            295198 non-null  int64         
 10  compile_error            295198 non-null  int64         
 11  syntax_error             295198 non-null  int64         
 12  pending         

### Tạo cột đặc trưng TBTL dựa theo đề cương môn IT001

In [18]:
mask_has_all_scores = df_merged[['TH', 'diemqt', 'CK']].notna().all(axis=1)
df_merged.loc[mask_has_all_scores, 'TBTL'] = (
    df_merged['TH'] * 0.3 +
    df_merged['diemqt'] * 0.2 +
    df_merged['CK'] * 0.5
)

In [19]:
df_clean = df_merged.copy()

In [20]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   assignment_id            295198 non-null  int64         
 1   problem_id               295198 non-null  int64         
 2   mssv                     295198 non-null  object        
 3   is_final                 295198 non-null  int64         
 4   score                    295198 non-null  int64         
 5   late_coef                295198 non-null  int64         
 6   lang_id                  295198 non-null  int64         
 7   submit_time              295198 non-null  datetime64[ns]
 8   judge_time               295198 non-null  datetime64[ns]
 9   code_runnable            295198 non-null  int64         
 10  compile_error            295198 non-null  int64         
 11  syntax_error             295198 non-null  int64         
 12  pending         

### Dự đoán CK

In [21]:
df_ck = df_clean[df_clean['CK'].notna()]
df_ck_missing = df_clean[df_clean['CK'].isna()]

In [22]:
features = [
    'assignment_id', 'score', 'late_coef', 'lang_id', 'testcase_passed_percent',
    'n_tests', 'n_failed', 'failed_ratio', 'avg_time', 'max_time', 
    'avg_mem', 'max_mem', 'code_runnable', 'compile_error', 'syntax_error'
]


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X = df_ck[features]
y = df_ck['CK']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_ck = XGBRegressor(n_estimators=100, random_state=42)
model_ck.fit(X_train, y_train)

y_pred = model_ck.predict(X_val)
print("r2:", r2_score(y_val, y_pred))


r2: 0.23092528436346615


In [24]:
df_clean.loc[df_clean['CK'].isna(), 'CK_pred'] = model_ck.predict(df_ck_missing[features])

### Dự đoán QT

In [25]:
df_qt = df_clean[df_clean['diemqt'].notna()]
df_qt_missing = df_clean[df_clean['diemqt'].isna()]

In [26]:
X = df_qt[features]
y = df_qt['diemqt']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_qt = RandomForestRegressor(n_estimators=100, random_state=42)
model_qt.fit(X_train, y_train)

y_pred = model_qt.predict(X_val)
print("r2:", r2_score(y_val, y_pred))

r2: 0.47325663259719175


In [27]:
df_clean.loc[df_clean['diemqt'].isna(), 'qt_pred'] = model_qt.predict(df_qt_missing[features])

### Dự đoán TH

In [28]:
df_th = df_clean[df_clean['TH'].notna()]
df_th_missing = df_clean[df_clean['TH'].isna()]

In [29]:
X = df_th[features]
y = df_th['TH']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model_th = XGBRegressor(n_estimators=100, random_state=42)
model_th.fit(X_train, y_train)

y_pred = model_th.predict(X_val)
print("r2:", r2_score(y_val, y_pred))

r2: 0.28172187038919594


In [30]:
df_clean.loc[df_clean['TH'].isna(), 'TH_pred'] = model_th.predict(df_th_missing[features])

In [31]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 28 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   assignment_id            295198 non-null  int64         
 1   problem_id               295198 non-null  int64         
 2   mssv                     295198 non-null  object        
 3   is_final                 295198 non-null  int64         
 4   score                    295198 non-null  int64         
 5   late_coef                295198 non-null  int64         
 6   lang_id                  295198 non-null  int64         
 7   submit_time              295198 non-null  datetime64[ns]
 8   judge_time               295198 non-null  datetime64[ns]
 9   code_runnable            295198 non-null  int64         
 10  compile_error            295198 non-null  int64         
 11  syntax_error             295198 non-null  int64         
 12  pending         

In [32]:
df_clean['CK'] = df_clean['CK'].fillna(df_clean['CK_pred'])
df_clean['diemqt'] = df_clean['diemqt'].fillna(df_clean['qt_pred'])
df_clean['TH'] = df_clean['TH'].fillna(df_clean['TH_pred'])

In [33]:
df_clean = df_clean.drop(columns=['CK_pred', 'qt_pred', 'TH_pred'])

In [34]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 295198 entries, 0 to 295197
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   assignment_id            295198 non-null  int64         
 1   problem_id               295198 non-null  int64         
 2   mssv                     295198 non-null  object        
 3   is_final                 295198 non-null  int64         
 4   score                    295198 non-null  int64         
 5   late_coef                295198 non-null  int64         
 6   lang_id                  295198 non-null  int64         
 7   submit_time              295198 non-null  datetime64[ns]
 8   judge_time               295198 non-null  datetime64[ns]
 9   code_runnable            295198 non-null  int64         
 10  compile_error            295198 non-null  int64         
 11  syntax_error             295198 non-null  int64         
 12  pending         

### Train / test

In [35]:
df_train = df_clean[df_clean['TBTL'].notna()]
df_test = df_clean[df_clean['TBTL'].isna()]

features = [
    'CK', 'diemqt', 'TH',
    'score', 'testcase_passed_percent', 'failed_ratio',
    'avg_time', 'max_time', 'avg_mem', 'max_mem',
    'code_runnable', 'compile_error', 'syntax_error',
    'late_coef', 'lang_id'
]

X = df_train[features]
y = df_train['TBTL']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)

print("R² score:", r2_score(y_val, y_pred))

R² score: 0.9999998625462923


### Dự đoán

In [36]:
X_test = df_test[features]
df_clean.loc[df_clean['TBTL'].isna(), 'TBTL'] = model.predict(X_test)

In [37]:
df_clean.drop_duplicates()

,assignment_id,problem_id,mssv,is_final,score,late_coef,lang_id,submit_time,judge_time,code_runnable,...,n_failed,failed_ratio,avg_time,max_time,avg_mem,max_mem,CK,diemqt,TH,TBTL
0,116,208,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,0,100,1,2024-10-09 08:02:04,2024-10-09 08:06:58,1,...,0.0,0.0,0.0,0.0,0.0,0.0,4.101735,4.526822,6.299403,5.3375
1,116,208,ed9eaeb6a707f50154024b24d7efcb874a9795dd,0,0,100,1,2024-10-09 08:04:41,2024-10-09 08:04:51,1,...,0.0,0.0,0.0,0.0,0.0,0.0,4.101735,4.526822,6.299403,5.3375
2,116,208,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,10000,100,1,2024-10-09 08:06:49,2024-10-09 08:06:58,1,...,0.0,0.0,0.0,0.0,0.0,0.0,4.946250,6.584717,7.364905,6.1705
3,116,335,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,10000,100,1,2024-10-09 08:47:52,2024-10-09 08:48:01,1,...,0.0,0.0,0.0,0.0,0.0,0.0,4.946250,6.584717,7.364905,6.1705
4,116,212,ed9eaeb6a707f50154024b24d7efcb874a9795dd,1,10000,100,1,2024-10-09 09:19:35,2024-10-09 09:19:45,1,...,0.0,0.0,0.0,0.0,0.0,0.0,4.946250,6.584717,7.364905,6.1705
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295193,76,97,232cce96362898f08e9150ba244adaf2d6583ab2,1,10000,100,1,2024-01-15 16:03:43,2024-01-15 16:03:53,1,...,0.0,0.0,0.0,0.0,0.0,0.0,6.000000,7.500000,9.000000,7.2000
295194,76,378,232cce96362898f08e9150ba244adaf2d6583ab2,0,0,100,1,2024-01-15 16:04:07,2024-01-15 16:05:08,0,...,0.0,0.0,0.0,0.0,0.0,0.0,6.000000,7.500000,9.000000,7.2000
295195,76,378,232cce96362898f08e9150ba244adaf2d6583ab2,1,10000,100,1,2024-01-15 16:04:58,2024-01-15 16:05:08,1,...,0.0,0.0,0.0,0.0,0.0,0.0,6.000000,7.500000,9.000000,7.2000
295196,76,242,232cce96362898f08e9150ba244adaf2d6583ab2,1,10000,100,1,2024-01-15 16:05:13,2024-01-15 16:05:22,1,...,0.0,0.0,0.0,0.0,0.0,0.0,6.000000,7.500000,9.000000,7.2000


### Nộp kết quả

In [38]:
submission = df_clean[['mssv', 'TBTL']].drop_duplicates()

submission = submission.groupby('mssv', as_index=False)['TBTL'].mean()

In [39]:
submission[['mssv', 'TBTL']].to_csv('result.csv', index=False, header=False, encoding='utf-8')

### Lưu bộ siêu tham số tốt nhất

In [40]:
joblib.dump(model, 'best_model.pkl')

['best_model.pkl']

### In ra bộ siêu tham số tốt nhất

In [41]:
model = joblib.load('best_model.pkl')

print(model)

# Xem các tham số mô hình
print(model.get_params())

RandomForestRegressor(random_state=42)
{'bootstrap': True, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': None, 'max_features': 1.0, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 100, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
